In [ ]:
import re
import geopandas as gpd
import pandas as pd
from shapely.ops import unary_union, polygonize

# --------------------------------------------------
# CONFIG
# --------------------------------------------------
# The state you want to process (e.g., "CT", "RI", "MA")
TARGET_STATE = "PA"

# Input national shapefile
INPUT_SHP = "/Users/kuba/Downloads/electric-retail-service-territories-shapefile/Electric_Retail_Service_Territories.shp"

# Output final exact overlap cells
OUTPUT_CELLS = f"/Users/kuba/PycharmProjects/grid-research/data/OUTPUT_CELLS_{TARGET_STATE}.geojson"

NAME_COL = "NAME"
STATE_COL = "STATE" # usually 'STATE' in HIFLD


# --------------------------------------------------
# HELPERS
# --------------------------------------------------
def choose_local_projected_crs(gdf):
    """
    If input CRS is geographic, choose a local UTM CRS based on layer centroid.
    This makes polygon operations / area much more reliable.
    """
    if gdf.crs is None:
        raise ValueError("Input has no CRS.")

    if not gdf.crs.is_geographic:
        return gdf.crs

    gdf_wgs84 = gdf.to_crs(4326)
    centroid = gdf_wgs84.unary_union.centroid
    lon, lat = centroid.x, centroid.y

    zone = int((lon + 180) // 6) + 1
    epsg = 32600 + zone if lat >= 0 else 32700 + zone
    return f"EPSG:{epsg}"


def clean_name(name):
    """Light normalization only."""
    if pd.isna(name):
        return ""
    name = str(name).upper().strip()
    name = re.sub(r"\s+", " ", name)
    return name


def parse_utilities(name):
    """Generic parser for HIFLD-style territory names."""
    name = clean_name(name)
    if not name:
        return []

    name = re.sub(r"\bJOINT\b", "", name).strip()
    name = re.sub(r"\s+", " ", name)
    parts = re.split(r"\s*/\s*", name)
    parts = [re.sub(r"\s+", " ", p).strip(" -;,") for p in parts]
    parts = [p for p in parts if p]

    return sorted(set(parts))


# --------------------------------------------------
# MAIN PIPELINE
# --------------------------------------------------
def build_exact_overlap_cells_from_gdf(gdf, name_col="NAME", output_cells="exact_overlap_cells.geojson"):
    """
    Modified to accept a GeoDataFrame directly instead of reading a file path.
    Only outputs the overlap_cells geojson as requested.
    """
    if name_col not in gdf.columns:
        raise ValueError(f"Column '{name_col}' not found. Columns are: {list(gdf.columns)}")

    # keep only non-empty geometries
    gdf = gdf[gdf.geometry.notna() & (~gdf.geometry.is_empty)].copy()

    if len(gdf) == 0:
        raise ValueError("No non-empty geometries found for this state.")

    original_crs = gdf.crs
    projected_crs = choose_local_projected_crs(gdf)
    gdf = gdf.to_crs(projected_crs)

    # basic geometry repair
    gdf["geometry"] = gdf.buffer(0)
    gdf = gdf[gdf.geometry.notna() & (~gdf.geometry.is_empty)].copy()

    # parse utility lists
    gdf["utility_list"] = gdf[name_col].apply(parse_utilities)
    gdf = gdf[gdf["utility_list"].map(len) > 0].copy()

    if len(gdf) == 0:
        raise ValueError("No rows left after parsing utility names.")

    # explode rows
    exploded = gdf[[name_col, "geometry", "utility_list"]].explode("utility_list")
    exploded = exploded.rename(columns={"utility_list": "utility"}).copy()

    # rebuild one footprint per utility
    footprints = exploded[["utility", "geometry"]].dissolve(by="utility", as_index=False)
    footprints["geometry"] = footprints.buffer(0)
    footprints = footprints[footprints.geometry.notna() & (~footprints.geometry.is_empty)].copy()

    if len(footprints) == 0:
        raise ValueError("No utility footprints were reconstructed.")

    # union of all utility boundaries and polygonize
    boundary_union = unary_union([geom.boundary for geom in footprints.geometry])
    cells = list(polygonize(boundary_union))

    if len(cells) == 0:
        raise ValueError("Polygonize produced no cells.")

    coverage_union = unary_union(list(footprints.geometry))
    sindex = footprints.sindex

    rows = []
    for i, cell in enumerate(cells):
        pt = cell.representative_point()

        # skip outside the reconstructed utility coverage
        if not coverage_union.covers(pt):
            continue

        candidate_idx = list(sindex.query(pt, predicate="intersects"))
        if not candidate_idx:
            continue

        members = []
        for idx in candidate_idx:
            geom = footprints.geometry.iloc[idx]
            if geom.covers(pt):
                members.append(footprints["utility"].iloc[idx])

        members = sorted(set(members))
        if not members:
            continue

        rows.append(
            {
                "cell_id": i,
                "utilities": " | ".join(members),
                "n_utils": len(members),
                "geometry": cell,
            }
        )

    overlap_cells = gpd.GeoDataFrame(rows, geometry="geometry", crs=projected_crs)

    if len(overlap_cells) == 0:
        raise ValueError("No overlap cells were created.")

    # Convert back to original CRS and save
    if original_crs is not None:
        overlap_cells_out = overlap_cells.to_crs(original_crs)
    else:
        overlap_cells_out = overlap_cells

    overlap_cells_out.to_file(output_cells, driver="GeoJSON")

    print("\n--- Pipeline Complete ---")
    print(f"State processed: {TARGET_STATE}")
    print(f"Rows in input (non-empty): {len(gdf)}")
    print(f"Reconstructed utility footprints: {len(footprints)}")
    print(f"Exact overlap cells: {len(overlap_cells)}")
    print(f"Saved ONLY the output cells to: {output_cells}")

    return overlap_cells_out


# --------------------------------------------------
# RUN
# --------------------------------------------------
if __name__ == "__main__":
    print(f"Loading national utility data from shapefile...")
    national_gdf = gpd.read_file(INPUT_SHP)

    print(f"Filtering for state: {TARGET_STATE}...")
    state_gdf = national_gdf[national_gdf[STATE_COL] == TARGET_STATE].copy()

    if state_gdf.empty:
        print(f"Error: Could not find data for state '{TARGET_STATE}'. Check the state abbreviation.")
    else:
        print(f"Found {len(state_gdf)} utility geometries for {TARGET_STATE}. Running pipeline...")

        # Run the pipeline passing the filtered dataframe directly
        result_gdf = build_exact_overlap_cells_from_gdf(
            gdf=state_gdf,
            name_col=NAME_COL,
            output_cells=OUTPUT_CELLS
        )